# A4 — Colab Driver

**Goal**: produce two missing submission files that local 4 GB GPU cannot make:

- `results/t5_scr_test.{sql,pkl}` — T5 trained from scratch (Task 2)
- `results/test_test.{sql,pkl}` — Gemma prompting (Task 3)

`results/t5_ft_test.{sql,pkl}` is already in the repo (dev F1 ≈ 0.52, produced locally).

**Pre-flight**: Runtime → Change runtime type → **T4** (free) is enough; **A100** if you have Pro+.


## 1. Get the project into the Colab runtime

GitHub Classroom repos are private, so a plain `git clone` fails. Pick
**one** of the two cells below — run only that one, skip the other.


### 1a. Clone via GitHub Personal Access Token (fastest)


In [ ]:
# Generate a *classic* PAT at https://github.com/settings/tokens with
# the `repo` scope. The token is held in memory only and dropped after
# the clone — it is NOT persisted to disk.
import os, subprocess, getpass

OWNER = 'Cornell-Tech-CS5744-Spring-2026'
REPO  = 'a4-D-Tony2020'
PROJECT_DIR = f'/content/{REPO}'

if not os.path.exists(PROJECT_DIR):
    gh_token = getpass.getpass('GitHub PAT (repo scope): ')
    url = f'https://{gh_token}@github.com/{OWNER}/{REPO}.git'
    subprocess.run(['git', 'clone', url, PROJECT_DIR], check=True)
    del gh_token, url  # leave no token in the notebook state

os.chdir(PROJECT_DIR)
!git pull --quiet
!ls --color=never


### 1b. (Alternative) Mount Google Drive and use a copy already there


In [ ]:
# Use this branch if you uploaded the project folder to Drive at
# `MyDrive/A4/a4-D-Tony2020`. Edit `PROJECT_DIR` if your path differs.
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = '/content/drive/MyDrive/A4/a4-D-Tony2020'
assert os.path.exists(PROJECT_DIR), f'edit PROJECT_DIR — not found: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
!ls --color=never


In [ ]:
# Colab's base image already ships torch+CUDA; just add the project deps.
!pip install -q transformers==4.51.3 accelerate==0.29.3 bitsandbytes==0.43.1 \
    sentencepiece==0.2.0 tokenizers==0.21.0 nltk==3.8.1 wandb==0.15.10 tqdm==4.66.1
import torch, transformers
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 2. HuggingFace login (only required for Gemma)


In [ ]:
from huggingface_hub import login
import getpass

# Use a *classic Read* token (https://huggingface.co/settings/tokens).
# Fine-grained tokens need extra perms for gated repos.
# License must already be accepted at https://huggingface.co/google/gemma-3-1b-it
token = getpass.getpass('HF token: ')
login(token=token, add_to_git_credential=False)
print('logged in OK')


## 3. Cache GT dev records (one-off, ~40 s)


In [ ]:
!python cache_gt_records.py --split dev


## 4. FAST PATH — close the two missing submissions

Runs T5 from scratch + Gemma 1B 3-shot BM25-with-schema in sequence.
On a T4 this is roughly 3 h training + 30 min prompting. The batch mode
continues even if one config fails so you don't lose the night to a bug.


In [ ]:
!python colab_train.py --task batch --config \
    "t5:t5_scr_colab,prompting:gemma1b_k3_bm25_schema"


## 5. (Optional) bonus — top the local T5 ft baseline

iter 001 on a 4 GB local card hit dev_F1=0.5171. With Colab's 16 GB+ you
can afford bigger batches and beam search at decode time, which often
adds 1-3 F1 points. Skip if you only need the milestone.


In [ ]:
!python colab_train.py --task t5 --config t5_ft_colab


In [ ]:
# Aggressive 20-epoch finetune. Run only if the cheaper t5_ft_colab
# already beat the local baseline.
!python colab_train.py --task t5 --config t5_ft_colab_long


## 6. (Optional) prompting ablations for the report


In [ ]:
# Sweep three configs in one go: zero-shot, 3-shot random, 3-shot BM25
# (the schema-on variant was already covered in the FAST PATH).
!python colab_train.py --task batch --config \
    "prompting:gemma1b_k0,prompting:gemma1b_k3_random,prompting:gemma1b_k3_bm25"


## 7. (Optional, A100) — bigger Gemma

On a T4 this will work but generate slowly. On an A100 it's painless.


In [ ]:
!python colab_train.py --task prompting --config gemma27b_k3_bm25_schema_q4


## 8. Pick best per track and rename to leaderboard files


In [ ]:
!python make_submission.py


## 9. Dashboard + push results back to GitHub


In [ ]:
!python colab_train.py --task dashboard


In [ ]:
# Configure git author once per Colab session.
!git config user.email "your@email"
!git config user.name "Your Name"

# Verify the canonical submission files exist before pushing.
!ls -la results/t5_ft_test.sql results/t5_scr_test.sql results/test_test.sql 2>/dev/null
!ls -la records/t5_ft_test.pkl records/t5_scr_test.pkl records/test_test.pkl 2>/dev/null

# Stage the artefacts the leaderboard needs + the registry.
!git add results/ records/ experiments/registry.csv experiments/runs/
!git status -s


In [ ]:
# Commit + push. Edit the message if you want.
commit_msg = "Colab batch: t5_scr + gemma_1b prompting submissions"
!git commit -m "$commit_msg"
!git push origin main
